## **Noise modeling from magnitude bins per filter**

In [1]:
import numpy as np
from astropy.table import Table
from astropy.io import fits

#### ***(1) Computing 1-sigma depth limits***

Using reported 5-sigma depth limits per filter

In [2]:
# import table with 5-sigma depths per filter
sig5_tab = Table.read('table_5sigdepths.txt', format='ascii')

FileNotFoundError: [Errno 2] No such file or directory: 'table_5sigdepths.txt'

In [ ]:
# compute mJy fluxes for 5-sigma depths (mJy is units of SBIPIX model training and forward-modeling)
def mJy_flux(mags):
    'input magnitude 5-sigma depths as entire array, output mJy fluxes as entire array'
    return (3631/1e-6)*(10**(mags/(-2.5)))

sig5_mJy = mJy_flux(sig5_tab['PSF_total'])

# add to running table with 5-sigma depths
sig5_tab['PSF_total_mJy'] = sig5_mJy

In [ ]:
# compute 1-sigma mJy depth using pixel aperture correction
def mJy_flux_1sig(mJy_5sig):
    'input mJy 5-sigma depths as entire array, output mJy 1-sigma depths as entire array'
    sig1depth = mJy_5sig/5

    area_circ1 = ((0.1)**2)*np.pi # 1 pixel aperture area in arcsec^2
    area_pix = 0.03*0.03 # pixel area in arcsec^2
    npix_aperture = area_circ1/area_pix # number of pixels in 1 pixel aperture
    sig1depth_perpixel = sig1depth/np.sqrt(npix_aperture) # 3 pixel aperture correction

    return sig1depth_perpixel

sig1_mJy_perpixel = mJy_flux_1sig(sig5_tab['PSF_total_mJy'])

# add to running table with 5-sigma depths
sig5_tab['perpixel_mJy_1sig'] = sig1_mJy_perpixel

In [ ]:
# make any adjustments to filter selection and ordering here before saving
sig5_tab

In [ ]:
# save the 1-sigma mJy depth per pixel as a numpy array for use in SBIPIX model training and forward-modeling
np.save('background_noise_field.npy', sig1_mJy_perpixel)

#### ***(2) Constructing magnitude bins***

Uniform binning with 20 bins and necessary truncation of last brightest (and sparsest) bins

##### Compile all magnitudes and magnitude errors for field

In [ ]:
# import zero point and dust correction table (provided per filter) to apply later to magnitudes
zp_dustcorr_tab = Table.read('GOODSS_v095_zp-dustcorr.txt', format='ascii')

zp = zp_dustcorr_tab['zp']
dustcorr = zp_dustcorr_tab['dustcorr']

In [ ]:
# a quick function to pull filenames easily given the filter names recorded earlier
field = 'GOODSS' # change this to the field you are working on

def get_filenames(instrument, filtername, sci_or_err):
    'input instrument (nrc or acs) and filter name (lowercase) and science or error flag (sci or err), output filename'
    if instrument == 'nrc':
        if sci_or_err == 'sci':
            if filtername in ['f444w', 'f460m', 'f480m']:
                filename = f'{field.lower()}_{instrument}_{filtername}_{sci_or_err}.fits'
            else:
                filename = f'{field.lower()}_{instrument}_{filtername}_{sci_or_err}_matchf444w.fits'
        elif sci_or_err == 'err':
            filename = f'{field.lower()}_{instrument}_{filtername}_{sci_or_err}.fits'
    elif instrument == 'acs':
        if sci_or_err == 'sci':
            filename = f'{field.lower()}_{instrument}_{filtername}_{sci_or_err}_matchf444w.fits'
        elif sci_or_err == 'err':
            filename = f'{field.lower()}_{instrument}_{filtername}_{sci_or_err}.fits'
    return filename

In [ ]:
# identify all pixels in the field mosaic by using the segmentation map
path = '/Volumes/dataSSD/GOODSS/GOODSS/'
segmap = fits.open(path+'/segmap_goodss_v0.891.fits')
seg = segmap[0].data

# we impose a SNR>2.5 cut per pixel later, so select only those pixels here as well to be representative of our ultimate observational dataset
# the SNR cut is imposed on the F444W filter, so we select that one here
err_mosaic = fits.open(path+'data/'+get_filenames('nrc', 'f444w', 'err'))
err = err_mosaic[0].data
phot_mosaic = fits.open(path+'data/'+get_filenames('nrc', 'f444w', 'sci'))
photo = phot_mosaic[0].data

# select pixels that are part of a galaxy and meet the SNR>2.5 cut in F444W
# cut out SNR cut
mask = (seg!=0)# & (photo/err >= 2.5))
# get the indices of the pixels that meet the criteria, in flattened array
mask_idx = np.where(mask.ravel())[0]

In [ ]:
# save length of segmap to have number of pixels handy in array construction
npix = seg[mask].shape[0]

# make arrays to store magnitudes and magnitude errors for each filter, for each pixel
mags = {f: np.full(npix, np.nan) for f in sig5_tab['Filter']}
mags_err = {f: np.full(npix, np.nan) for f in sig5_tab['Filter']}

In [ ]:
# function to apply zero point and dust correction to fluxes
def flux_correction(flux_or_err, zp, dustcorr):
    'input nJy fluxes or errors, zero point, and dust correction, output corrected mJy fluxes or errors'
    corr = flux_or_err*zp*1.0e32*dustcorr
    mJy = corr * 1e-3  # Convert nJy to microJy
    return mJy

# functions to convert mJy fluxes and errors to magnitudes
def flux_to_mag(flux_mJy):
    'input mJy fluxes, output magnitudes'
    mag = -2.5 * np.log10(flux_mJy * 1e-6 / 3631)
    return mag

def err_to_mag(err_mJy, flux_mJy):
    'input mJy flux errors, output magnitude errors'
    mag_err = (2.5 * err_mJy)/(flux_mJy * np.log(10))
    return mag_err

In [ ]:
# function to compile magnitudes and magnitude errors for each filter and pixel into preconstructed arrays
def compile_magnitudes(filter, instrument, validpix_mask):
    'input filter name and valid pixel mask, output magnitudes and magnitude errors for each pixel in the filter'
    # get the science and error mosaics for the filter
    sci_mosaic = fits.open(path+'data/'+get_filenames(instrument, filter, 'sci'))
    err_mosaic = fits.open(path+'data/'+get_filenames(instrument, filter, 'err'))

    # mask by the valid pixels determined earlier
    sci = sci_mosaic[0].data[validpix_mask]
    err = err_mosaic[0].data[validpix_mask]

    # impose additional restrictions on valid pixels: that flux is positive and that error and flux are finite
    # also impose cut on SNR>1 per pixel (remember the only other cut we imposed was SNR>2.5 in F444W specifically)
    # decided to cut this out ^
    xtra_mask  = (sci > 0) & np.isfinite(err) & np.isfinite(sci)# & (sci/err > 1)

    # apply zero point and dust correction to fluxes and errors
    flux_corr = flux_correction(sci[xtra_mask], zp[filter], dustcorr[filter])
    err_corr = flux_correction(err[xtra_mask], zp[filter], dustcorr[filter])

    # convert corrected fluxes and errors to magnitudes and magnitude errors
    mag = flux_to_mag(flux_corr)
    mag_err = err_to_mag(err_corr, flux_corr)

    mags[filter][xtra_mask] = mag
    mags_err[filter][xtra_mask] = mag_err